In [ ]:
from recipe import TrainValidationTest, DataCleaner

from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier, VotingClassifier, StackingClassifier, BaggingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, ParameterGrid, ParameterSampler, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

from tqdm import tqdm
import pandas as pd
import joblib
import numpy as np
from scipy.stats import randint, loguniform



In [6]:
def metrics(model, X, y):
    y_pred = model.predict(X)
    return {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y, y_pred, average='macro', zero_division=0),
    }

def get_candidates(grid):
    if isinstance(grid, RandomizedSearchCV):
        return list(ParameterSampler(grid.param_distributions,
                                     n_iter=grid.n_iter, random_state=grid.random_state))
    return list(ParameterGrid(grid.param_grid))

def best_results(grids, names, X_train, y_train, X_valid, y_valid):
    rows = []
    for grid, name in zip(grids, names):
        best_score, best_params = -float('inf'), None

        for params in tqdm(get_candidates(grid), desc=name):
            model = clone(grid.estimator).set_params(**params)
            score = cross_val_score(model, X_train, y_train, cv=grid.cv,
                                    scoring=grid.scoring, n_jobs=grid.n_jobs).mean()
            if score > best_score:
                best_score, best_params = score, params

        best_model = clone(grid.estimator).set_params(**best_params).fit(X_train, y_train)
        y_pred_valid = best_model.predict(X_valid)
        if grid.scoring == 'accuracy':
            valid_score = accuracy_score(y_valid, y_pred_valid)
        else:
            valid_score = precision_score(y_valid, y_pred_valid, average='macro', zero_division=0)

        rows.append({
            'model': name,
            'params': best_params,
            'cv_score': best_score,
            'valid_score': valid_score,
            'search': best_model,
        })
        tqdm.write(f'{name}: CV {best_score:.3f}, valid {valid_score:.3f}, {best_params}\n')

    return pd.DataFrame(rows)

def choose(grids, names, X_train, y_train, X_valid, y_valid):
    result = best_results(grids, names, X_train, y_train, X_valid, y_valid)
    best = result.loc[result['cv_score'].idxmax()]
    print(f'best estimator: {best["model"]}')
    return best, result

In [7]:
raw = pd.read_csv('data/epi_r_processed.csv')
df = DataCleaner(save_to_csv=True).fit_transform(raw)

[DataCleaner] After drop duplicates. Shape = (18235, 351)
[DataCleaner] After drop duplicates. Shape = (18235, 351)
[DataCleaner] Before memory optimization
<class 'pandas.DataFrame'>
RangeIndex: 18235 entries, 0 to 18234
Columns: 351 entries, title to turkey
dtypes: float64(5), int64(345), str(1)
memory usage: 48.8 MB
None

[DataCleaner] After memory optimization
<class 'pandas.DataFrame'>
RangeIndex: 18235 entries, 0 to 18234
Columns: 351 entries, title to turkey
dtypes: float64(5), int8(345), str(1)
memory usage: 6.8 MB
None

[DataCleaner] Final ingredients count: 345
[DataCleaner] Набор данных сохранен ((18235, 351)) сохранен в csv data/epi_r_processed.csv


In [8]:
# df.isna().sum()[df.isna().sum() > 0] 

In [9]:
df['rating'] = round(df['rating'])

X = df.drop(columns=['rating', 'title'])
y = df['rating']
cols = ['protein', 'calories', 'fat', 'sodium']
X['nutrition_no_data'] = X[cols].isna().any(axis=1).astype(int)
X[cols] = X[cols].fillna(0)

X_train, X_valid, X_test, y_train, y_valid, y_test = TrainValidationTest().get_datasets(X, y)
scaler = StandardScaler().fit(X_train)
X_train, X_valid, X_test = (scaler.transform(X_train), scaler.transform(X_valid), scaler.transform(X_test))


In [10]:
cv = StratifiedKFold(n_splits=5, random_state=21, shuffle=True)

#если что, гриды взяты по основе с epi_r_regression
grids = [
    GridSearchCV(LogisticRegression(penalty=None, max_iter=2000), {},
                 cv=cv, scoring="accuracy", n_jobs=-1),

    GridSearchCV(RidgeClassifier(random_state=21),
                 {"alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                 cv=cv, scoring="accuracy", n_jobs=-1),

    GridSearchCV(SGDClassifier(penalty="l1", random_state=21, max_iter=2000), {
                 "alpha": [1e-5, 1e-4, 1e-3, 1e-2],
                 "loss": ["hinge", "log_loss"],
             }, cv=cv, scoring="accuracy", n_jobs=-1),

    GridSearchCV(
        RandomForestClassifier(n_estimators=300, random_state=21),
        {
            "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2"],
        },
        cv=cv, scoring="accuracy", n_jobs=-1),

    RandomizedSearchCV(HistGradientBoostingClassifier(random_state=21), {
                 "max_iter": [200, 300, 500],
                 "learning_rate": loguniform(0.02, 0.3),
                 "max_depth": [3, 5, None],
                 "min_samples_leaf": randint(10, 60),
             }, n_iter=20, cv=cv, scoring="accuracy", n_jobs=-1, random_state=21),
]

names = ['logreg', 'ridge', 'sgd', 'random forest', 'hist boosting']

best, result = choose(grids, names, X_train, y_train, X_valid, y_valid)
display(result.drop(columns='search').sort_values('valid_score', ascending=False))

best_model = best['search']
print('test accuracy:', accuracy_score(y_test, best_model.predict(X_test)))

logreg: 100%|██████████| 1/1 [00:04<00:00,  4.03s/it]
c:\Users\Янки\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logreg: CV 0.665, valid 0.673, {}



ridge: 100%|██████████| 6/6 [00:06<00:00,  1.02s/it]


ridge: CV 0.673, valid 0.679, {'alpha': 1000}



sgd: 100%|██████████| 8/8 [05:26<00:00, 40.79s/it]


sgd: CV 0.670, valid 0.676, {'alpha': 0.01, 'loss': 'log_loss'}



... (2 outputs left out)

hist boosting: 100%|██████████| 20/20 [20:38<00:00, 61.92s/it]


hist boosting: CV 0.674, valid 0.680, {'learning_rate': np.float64(0.022056823203410247), 'max_depth': 3, 'max_iter': 500, 'min_samples_leaf': 59}

best estimator: random forest


,model,params,cv_score,valid_score
3,random forest,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.675493,0.681974
4,hist boosting,"{'learning_rate': 0.022056823203410247, 'max_d...",0.673950,0.679918
1,ridge,{'alpha': 1000},0.672579,0.678890
2,sgd,"{'alpha': 0.01, 'loss': 'log_loss'}",0.670437,0.676491
0,logreg,{},0.665124,0.672721


test accuracy: 0.672881820674527


In [11]:
common = y_train.mode()[0]
y_pred = [common] * len(y_valid)
print('naive accuracy:', accuracy_score(y_valid, y_pred))

print(y_valid.value_counts())
print(y_train.value_counts())
print(y.value_counts())

naive accuracy: 0.6644962302947224
rating
4.0    1939
5.0     381
0.0     264
3.0     217
2.0      94
1.0      23
Name: count, dtype: int64
rating
4.0    7753
5.0    1521
0.0    1058
3.0     868
2.0     377
1.0      93
Name: count, dtype: int64
rating
4.0    12115
5.0     2378
0.0     1652
3.0     1356
2.0      589
1.0      145
Name: count, dtype: int64


In [13]:
rating_revamped = pd.cut(df['rating'], bins=[0, 1, 3, 5], labels=['bad', 'so-so', 'great'], include_lowest=True)
y = rating_revamped.cat.codes

X_train, X_valid, X_test, y_train, y_valid, y_test = TrainValidationTest().get_datasets(X, y)
scaler = StandardScaler().fit(X_train)
X_train, X_valid, X_test = (scaler.transform(X_train), scaler.transform(X_valid), scaler.transform(X_test))

In [14]:
best, result = choose(grids, names, X_train, y_train, X_valid, y_valid)
display(result.drop(columns='search').sort_values('valid_score', ascending=False))

best_model = best['search']
print('test accuracy:', accuracy_score(y_test, best_model.predict(X_test)))

common = y_train.mode()[0]
y_pred = [common] * len(y_valid)
print('naive accuracy:', accuracy_score(y_valid, y_pred))

print(y_valid.value_counts())
print(y_train.value_counts())
print(y.value_counts())

logreg: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]
c:\Users\Янки\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logreg: CV 0.797, valid 0.799, {}



ridge: 100%|██████████| 6/6 [00:01<00:00,  3.31it/s]


ridge: CV 0.802, valid 0.800, {'alpha': 1000}



sgd: 100%|██████████| 8/8 [03:38<00:00, 27.31s/it]


sgd: CV 0.801, valid 0.798, {'alpha': 0.01, 'loss': 'hinge'}



... (2 outputs left out)

hist boosting: 100%|██████████| 20/20 [10:12<00:00, 30.64s/it]


hist boosting: CV 0.802, valid 0.801, {'learning_rate': np.float64(0.022056823203410247), 'max_depth': 3, 'max_iter': 500, 'min_samples_leaf': 59}

best estimator: random forest


,model,params,cv_score,valid_score
3,random forest,"{'max_depth': 30, 'max_features': 'sqrt', 'min...",0.802656,0.800891
4,hist boosting,"{'learning_rate': 0.022056823203410247, 'max_d...",0.801799,0.800548
1,ridge,{'alpha': 1000},0.802228,0.799863
0,logreg,{},0.796829,0.799178
2,sgd,"{'alpha': 0.01, 'loss': 'hinge'}",0.800686,0.798492


test accuracy: 0.8025774609267892
naive accuracy: 0.7947224126113777
2    2319
1     311
0     288
Name: count, dtype: int64
2    9275
1    1245
0    1150
Name: count, dtype: int64
2    14493
1     1945
0     1797
Name: count, dtype: int64


In [15]:
grids_prec = [clone(g).set_params(scoring='precision_macro') for g in grids]

best_p, result_p = choose(grids_prec, names, X_train, y_train, X_valid, y_valid)
display(result_p.drop(columns='search').sort_values('cv_score', ascending=False))

best_model = best_p['search']
print('test precision_macro:', precision_score(y_test, best_model.predict(X_test), average='macro', zero_division=0))

logreg: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]
c:\Users\Янки\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logreg: CV 0.504, valid 0.491, {}



ridge: 100%|██████████| 6/6 [00:01<00:00,  3.08it/s]


ridge: CV 0.542, valid 0.582, {'alpha': 1}



sgd: 100%|██████████| 8/8 [03:24<00:00, 25.52s/it]


sgd: CV 0.519, valid 0.516, {'alpha': 0.001, 'loss': 'log_loss'}



... (2 outputs left out)

hist boosting: 100%|██████████| 20/20 [10:20<00:00, 31.02s/it]


hist boosting: CV 0.540, valid 0.492, {'learning_rate': np.float64(0.06953396908776242), 'max_depth': 3, 'max_iter': 300, 'min_samples_leaf': 15}

best estimator: random forest


,model,params,cv_score,valid_score
3,random forest,"{'max_depth': 20, 'max_features': 'log2', 'min...",0.571778,0.515657
1,ridge,{'alpha': 1},0.541844,0.581647
4,hist boosting,"{'learning_rate': 0.06953396908776242, 'max_de...",0.539970,0.492279
2,sgd,"{'alpha': 0.001, 'loss': 'log_loss'}",0.519472,0.516257
0,logreg,{},0.503691,0.490547


test precision_macro: 0.5155662540119211


In [16]:
ranked = result_p.sort_values('cv_score', ascending=False)
top = ranked[ranked['search'].apply(lambda m: hasattr(m, 'predict_proba'))].head(3)

models = {row['model'].replace(' ', '_'): row['search'] for _, row in top.iterrows()}
base_learners = list(models.items()) 

classes = next(iter(models.values())).classes_
probas = [m.predict_proba(X_valid) for m in models.values()]
preds = [m.predict(X_valid) for m in models.values()]

weights_grid = [None, [2, 1, 1], [1, 2, 1], [1, 1, 2], [3, 2, 1], [1, 2, 3]]

voting_results = []


In [17]:
for weights in weights_grid:
    w = weights or [1] * len(models)

    proba_sum = sum(wi * p for wi, p in zip(w, probas))
    soft_pred = classes[np.argmax(proba_sum, axis=1)]
    
    vote_sum = sum(wi * (p[:, None] == classes) for wi, p in zip(w, preds))
    hard_pred = classes[np.argmax(vote_sum, axis=1)]

    for voting, y_pred in [('soft', soft_pred), ('hard', hard_pred)]:
        voting_results.append({
            'voting': voting,
            'weights': weights,
            'accuracy': accuracy_score(y_valid, y_pred),
            'precision': precision_score(y_valid, y_pred, average='macro', zero_division=0),
        })

vote_res = (pd.DataFrame(voting_results)
              .sort_values(['precision', 'accuracy'], ascending=False)
              .reset_index(drop=True))
display(vote_res)

best_vote = vote_res.loc[0]
voting_best = VotingClassifier(
    estimators=base_learners,
    voting=best_vote['voting'],
    weights=best_vote['weights'],
).fit(X_train, y_train)

,voting,weights,accuracy,precision
0,hard,"[1, 1, 2]",0.798149,0.516257
1,hard,"[1, 2, 3]",0.798149,0.516257
2,soft,"[2, 1, 1]",0.800548,0.501503
3,hard,None,0.800548,0.500649
4,hard,"[2, 1, 1]",0.800548,0.500649
5,hard,"[3, 2, 1]",0.800548,0.500649
6,soft,"[1, 1, 2]",0.801919,0.499214
7,soft,"[3, 2, 1]",0.800206,0.498657
8,soft,"[1, 2, 3]",0.801919,0.496750
9,soft,"[1, 2, 1]",0.800891,0.494844


In [ ]:
stack_results = []

for n in tqdm([3, 5], desc='stacking'):
    for passthrough in [False, True]:
        stacker = StackingClassifier(
            estimators=base_learners,
            final_estimator=LogisticRegression(max_iter=2000),
            cv=StratifiedKFold(n_splits=n, shuffle=True, random_state=21),
            passthrough=passthrough,
            n_jobs=-1,
        ).fit(X_train, y_train)

        stack_results.append({
            'n_splits': n,
            'passthrough': passthrough,
            **metrics(stacker, X_valid, y_valid),
            'model': stacker,
        })

stack_res = (pd.DataFrame(stack_results)
               .sort_values(['precision', 'accuracy'], ascending=False)
               .reset_index(drop=True))
display(stack_res.drop(columns='model'))

best_stacker = stack_res.loc[0, 'model']

stacking:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
bag_params = {
    'n_estimators': [10, 20, 30, 40, 50, 75, 100],
    'random_state': [21]
}
bag_grid = GridSearchCV(BaggingClassifier(estimator=best_model), bag_params, scoring='precision_macro', refit=True, n_jobs=-1, verbose=1).fit(X_train, y_train)

# result = pd.DataFrame(grid.cv_results_)
# result = (result[['param_n_estimators',
#                   'mean_test_accuracy', 'mean_test_precision', 'std_test_accuracy']]
#                   .sort_values(['mean_test_accuracy', 'mean_test_precision'], ascending=False)
#                   .reset_index(drop=True))

# best = result.iloc[0]
# bagger = BaggingClassifier(estimator=best_model, random_state=21, n_estimators=best['param_n_estimators']).fit(X_train, y_train)
bagger = bag_grid.best_estimator_

In [ ]:
ensembles = {
    'voting': voting_best,
    'stacking': best_stacker,
    'bagger': bagger,
}

comparison = (pd.DataFrame({name: metrics(m, X_valid, y_valid) for name, m in ensembles.items()})
                .T.sort_values(['precision', 'accuracy'], ascending=False))
display(comparison)

best_name = comparison.index[0]
best_ensemble = ensembles[best_name]
print(f'best ensemble: {best_name}')

,accuracy,precision,recall
stacking,0.801919,0.70467,0.801919
voting,0.800891,0.70373,0.800891


best ensemble: stacking


In [ ]:
print('TEST')
for metric, value in metrics(best_ensemble, X_test, y_test).items():
    print(f'{metric}: {value:.5f}')

naive = [y_train.mode()[0]] * len(y_test)
print(f'naive accuracy: {accuracy_score(y_test, naive):.5f}')
print(f'naive precision_macro: {precision_score(y_test, naive, average="macro", zero_division=0):.5f}')

TEST
accuracy: 0.80395
precision: 0.70768
recall: 0.80395
naive accuracy: 0.79490


In [ ]:
test_score = accuracy_score(y_test, best_ensemble.predict(X_test))

joblib.dump(scaler, 'data/scaler_clf.sav')
joblib.dump(best_ensemble, f'data/{best_name}_clf_acc_{test_score:.5f}.sav')

['data/stacker_clf_acc_0.80395.sav']